<a href="https://colab.research.google.com/github/knutzin/Stored-Procedure---LAB-Activities/blob/main/atividade_BD_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB: Stored Procedures & Functions - Banco de Dados II
**Ambiente:** Google Colab + Neon PostgreSQL
**GRUPO:** Nicolas Mariano da Silva - RA00333111 | Pedro Henrique Isamu Yoshissaro- RA00330820

In [ ]:
# 1. Configuração e Conexão Segura
!pip install ipython-sql psycopg2-binary

import os
from google.colab import userdata

connection_string = userdata.get('NEON_DB_URL')
os.environ['DATABASE_URL'] = connection_string

%load_ext sql

%config SqlMagic.style = 'plain'
%config SqlMagic.autopandas = False

%sql $DATABASE_URL

## Setup: Criação das Tabelas e População de Dados

In [ ]:
%%sql
DROP TABLE IF EXISTS audit_logs, order_items, orders, products, customers CASCADE;

CREATE TABLE customers (customer_id SERIAL PRIMARY KEY, name VARCHAR(100), email VARCHAR(100), loyalty_points INTEGER DEFAULT 0);
CREATE TABLE products (product_id SERIAL PRIMARY KEY, name VARCHAR(200), price NUMERIC(10,2), stock INTEGER DEFAULT 0, category VARCHAR(50), expiry_date DATE);
CREATE TABLE orders (order_id SERIAL PRIMARY KEY, customer_id INTEGER REFERENCES customers(customer_id), order_date TIMESTAMP DEFAULT NOW(), total_amount NUMERIC(12,2), status VARCHAR(20));
CREATE TABLE order_items (order_item_id SERIAL PRIMARY KEY, order_id INTEGER REFERENCES orders(order_id), product_id INTEGER REFERENCES products(product_id), quantity INTEGER, unit_price NUMERIC(10,2));
CREATE TABLE audit_logs (log_id SERIAL PRIMARY KEY, action VARCHAR(50), details TEXT, executed_at TIMESTAMP DEFAULT NOW());

-- Inserção de Clientes
INSERT INTO customers (name, email, loyalty_points) VALUES
('Alice Johnson', 'alice.j@email.com', 1250), ('Bob Smith', 'bob.s@email.com', 450), ('Carol Williams', 'carol.w@email.com', 890), ('David Brown', 'david.b@email.com', 2100), ('Emma Davis', 'emma.d@email.com', 320), ('Frank Wilson', 'frank.w@email.com', 675), ('Grace Taylor', 'grace.t@email.com', 1540), ('Henry Moore', 'henry.m@email.com', 80), ('Isabella Thomas', 'isabella.t@email.com', 980), ('Jack Anderson', 'jack.a@email.com', 1340), ('Karen Martinez', 'karen.m@email.com', 560), ('Liam Garcia', 'liam.g@email.com', 1870), ('Mia Rodriguez', 'mia.r@email.com', 420), ('Noah Lee', 'noah.l@email.com', 760), ('Olivia Walker', 'olivia.w@email.com', 1120), ('Paul Hall', 'paul.h@email.com', 295), ('Quinn Allen', 'quinn.a@email.com', 1430), ('Rachel Young', 'rachel.y@email.com', 630), ('Samuel King', 'samuel.k@email.com', 2050), ('Tina Wright', 'tina.w@email.com', 890);

-- Inserção de Produtos
INSERT INTO products (name, price, stock, category, expiry_date) VALUES
('Wireless Headphones', 89.99, 150, 'Electronics', '2027-12-31'), ('Smartphone Case', 19.99, 300, 'Accessories', '2028-06-30'), ('Organic Coffee Beans', 14.50, 80, 'Groceries', '2026-10-15'), ('Yoga Mat', 29.99, 120, 'Sports', '2029-01-01'), ('Laptop Backpack', 49.99, 95, 'Fashion', NULL), ('Protein Powder', 39.99, 60, 'Health', '2026-08-20'), ('Bluetooth Speaker', 59.99, 200, 'Electronics', '2027-11-30'), ('Running Shoes', 79.99, 45, 'Sports', NULL), ('Instant Noodles Pack', 5.99, 500, 'Groceries', '2026-09-10'), ('Wireless Mouse', 24.99, 180, 'Electronics', '2028-03-15'), ('Denim Jeans', 59.99, 75, 'Fashion', NULL), ('Vitamin D Supplement', 12.99, 250, 'Health', '2027-05-01'), ('Gaming Keyboard', 69.99, 110, 'Electronics', '2028-12-31'), ('Canned Tuna', 3.49, 400, 'Groceries', '2027-02-28'), ('Dumbbell Set', 45.00, 30, 'Sports', NULL), ('Sunglasses', 34.99, 90, 'Fashion', '2029-06-30'), ('Energy Bars (Box)', 18.99, 150, 'Groceries', '2026-11-20'), ('USB-C Cable', 9.99, 350, 'Accessories', '2028-07-15'), ('Fitness Tracker', 129.99, 55, 'Electronics', NULL), ('Face Mask Pack', 7.99, 280, 'Health', '2026-12-31');


# Atividade 1: Price Adjustment

In [ ]:

%%sql
DROP PROCEDURE IF EXISTS update_product_prices(numeric);
CREATE OR REPLACE PROCEDURE update_product_prices(p_percentage NUMERIC) AS $$
DECLARE
    v_rows_affected INTEGER;
BEGIN
    UPDATE products
    SET price = CASE
        WHEN (price * (1 + p_percentage / 100)) < 0 THEN 0
        ELSE (price * (1 + p_percentage / 100))
    END;

    GET DIAGNOSTICS v_rows_affected = ROW_COUNT;

    RAISE NOTICE 'Atualização concluída. Total de produtos atualizados: %', v_rows_affected;
END;
$$ LANGUAGE plpgsql;

### Teste atividade 1

In [ ]:
%%sql
CALL update_product_prices(-50);


# Atividade 2: Category Discount

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS apply_category_discount(text, numeric);

CREATE OR REPLACE PROCEDURE apply_category_discount(
    p_category_name TEXT,
    p_discount_percentage NUMERIC
) AS $$
DECLARE
    v_rows_affected INTEGER;
BEGIN
    UPDATE products
    SET price = price * (1 - p_discount_percentage / 100)
    WHERE category = p_category_name;

    GET DIAGNOSTICS v_rows_affected = ROW_COUNT;

    INSERT INTO audit_logs (action, details)
    VALUES (
        'CATEGORY_DISCOUNT',
        'Desconto de ' || p_discount_percentage || '% em ' || p_category_name || '. Itens: ' || v_rows_affected
    );

    RAISE NOTICE 'Sucesso: % produtos atualizados.', v_rows_affected;
END;
$$ LANGUAGE plpgsql;

### Teste atividade 2

In [ ]:
%%sql
CALL apply_category_discount('Electronics', 15);

# Atividade 3: Order Processing

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS process_order(int, json);

CREATE OR REPLACE PROCEDURE process_order(
    p_customer_id INT,
    p_items JSON
) AS $$
DECLARE
    v_order_id INT;
    v_total_sum NUMERIC := 0;
    v_item RECORD;
BEGIN

    INSERT INTO orders (customer_id, total_amount, status)
    VALUES (p_customer_id, 0, 'pending')
    RETURNING order_id INTO v_order_id;

    FOR v_item IN SELECT * FROM json_array_elements(p_items)
    LOOP
        DECLARE
            v_prod_id INT := (v_item.value->>'product_id')::INT;
            v_qty INT := (v_item.value->>'quantity')::INT;
            v_price NUMERIC;
        BEGIN

            SELECT price INTO v_price FROM products WHERE product_id = v_prod_id;


            INSERT INTO order_items (order_id, product_id, quantity, unit_price)
            VALUES (v_order_id, v_prod_id, v_qty, v_price);


            UPDATE products
            SET stock = stock - v_qty
            WHERE product_id = v_prod_id;


            v_total_sum := v_total_sum + (v_price * v_qty);
        END;
    END LOOP;


    UPDATE orders
    SET total_amount = v_total_sum, status = 'completed'
    WHERE order_id = v_order_id;


    RAISE NOTICE 'Pedido #% processado com sucesso. Total: %', v_order_id, v_total_sum;

EXCEPTION
    WHEN OTHERS THEN
        RAISE EXCEPTION 'Erro no processamento. Transação revertida (Rollback). Erro: %', SQLERRM;
END;
$$ LANGUAGE plpgsql;

### Teste atividade 3

In [ ]:
%%sql
CALL process_order(2, '[{"product_id"\:1, "quantity"\:2}, {"product_id"\:3, "quantity"\:1}]'::json);

# Atividade 4: Low Stock Alert

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS notify_low_stock();

CREATE OR REPLACE PROCEDURE notify_low_stock() AS $$
DECLARE
    v_p_list TEXT;
BEGIN
    SELECT string_agg(name || ' (Estoque: ' || stock || ')', ', ')
    INTO v_p_list
    FROM products
    WHERE stock < 10;

    IF v_p_list IS NOT NULL THEN
        -- Removi a coluna created_at e o NOW() aqui:
        INSERT INTO audit_logs (action, details)
        VALUES ('LOW_STOCK_ALERT', 'Produtos com estoque crítico: ' || v_p_list);

        RAISE NOTICE 'ALERTA: Itens com estoque baixo: %', v_p_list;
    ELSE
        RAISE NOTICE 'Tudo certo! Nenhum produto com estoque baixo.';
    END IF;
END;
$$ LANGUAGE plpgsql;

### Atividade 5: Final Price Function

In [ ]:
%%sql
DROP FUNCTION IF EXISTS calculate_final_price(int, int, int);

CREATE OR REPLACE FUNCTION calculate_final_price(
    p_product_id INTEGER,
    p_quantity INTEGER,
    p_customer_id INTEGER
)
RETURNS NUMERIC AS $$
DECLARE
    v_unit_price NUMERIC;
    v_points INTEGER;
    v_final_price NUMERIC;
BEGIN

    SELECT price INTO v_unit_price FROM products WHERE product_id = p_product_id;


    SELECT loyalty_points INTO v_points FROM customers WHERE customer_id = p_customer_id;


    IF v_unit_price IS NULL THEN RETURN 0; END IF;


    v_final_price := (v_unit_price * p_quantity) * 1.10;


    IF COALESCE(v_points, 0) > 500 THEN
        v_final_price := v_final_price * 0.95;
    END IF;


    RETURN ROUND(v_final_price, 2);
END;
$$ LANGUAGE plpgsql;

### Atividade 6: Top Selling Products

In [ ]:
%%sql
DROP FUNCTION IF EXISTS top_selling_products(int);

CREATE OR REPLACE FUNCTION top_selling_products(p_days INT)
RETURNS TABLE(product_name TEXT, total_quantity_sold BIGINT) AS $$
BEGIN
    RETURN QUERY
    SELECT
        p.name::TEXT, -- Garante que é TEXT
        SUM(oi.quantity)::BIGINT -- Garante que é BIGINT para bater com o RETURNS TABLE
    FROM products p
    JOIN order_items oi ON p.product_id = oi.product_id
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_date >= (CURRENT_DATE - p_days)
    GROUP BY p.name
    ORDER BY SUM(oi.quantity) DESC
    LIMIT 10;
END;
$$ LANGUAGE plpgsql;

### Atividade 7: Customer Lifetime Value (LTV)

In [ ]:
%%sql
DROP FUNCTION IF EXISTS customer_ltv(int);

CREATE OR REPLACE FUNCTION customer_ltv(p_customer_id INT)
RETURNS NUMERIC AS $$
DECLARE
    v_total NUMERIC;
BEGIN

    SELECT COALESCE(SUM(total_amount), 0)
    INTO v_total
    FROM orders
    WHERE customer_id = p_customer_id
      AND status = 'completed';

    RETURN v_total;
END;
$$ LANGUAGE plpgsql;

### Atividade 8: Expiry Discount

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS apply_expiry_discount();


CREATE OR REPLACE PROCEDURE apply_expiry_discount() AS $$
BEGIN

    WITH updated_items AS (
        UPDATE products
        SET price = price * 0.80
        WHERE expiry_date BETWEEN CURRENT_DATE AND (CURRENT_DATE + 7)
        RETURNING product_id
    )
    INSERT INTO audit_logs (action, details)
    SELECT 'EXPIRY_DISCOUNT', 'Desconto aplicado em ' || COUNT(*) || ' produtos próximos do vencimento.'
    FROM updated_items;

    RAISE NOTICE 'Rotina de descontos semanais finalizada.';
END;
$$ LANGUAGE plpgsql;

-- nao consegui fazer com o CRON
-- SELECT cron.schedule('0 2 * * 1', 'CALL apply_expiry_discount()');

### Atividade 9: Log Cleanup

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS cleanup_logs();
DROP PROCEDURE IF EXISTS cleanup_old_audit_logs();

CREATE OR REPLACE PROCEDURE cleanup_old_audit_logs() AS $$
DECLARE
    v_deletados INT;
BEGIN
    DELETE FROM audit_logs
    WHERE created_at < NOW() - INTERVAL '1 year';

    GET DIAGNOSTICS v_deletados = ROW_COUNT;
    RAISE NOTICE 'Limpeza concluída: % registros antigos removidos.', v_deletados;
END;
$$ LANGUAGE plpgsql;

-- nao consegui fazer com o CRON
-- SELECT cron.schedule('0 3 1 * *', 'CALL cleanup_old_audit_logs()');

### Atividade 10: Daily Ecommerce Report

In [ ]:
%%sql
DROP PROCEDURE IF EXISTS daily_report();
DROP PROCEDURE IF EXISTS daily_ecommerce_report();

CREATE OR REPLACE PROCEDURE daily_ecommerce_report() AS $$
DECLARE
    v_total_sales NUMERIC := 0;
    v_customers_updated INT := 0;
    v_yesterday DATE := CURRENT_DATE - 1;
BEGIN
    SELECT COALESCE(SUM(total_amount), 0) INTO v_total_sales
    FROM orders
    WHERE order_date::DATE = v_yesterday;

    WITH updated_customers AS (
        UPDATE customers
        SET loyalty_points = loyalty_points + 10
        WHERE customer_id IN (
            SELECT DISTINCT customer_id FROM orders WHERE order_date::DATE = v_yesterday
        )
        RETURNING customer_id
    )
    SELECT COUNT(*) INTO v_customers_updated FROM updated_customers;

    INSERT INTO audit_logs (action, details, created_at)
    VALUES (
        'DAILY_REPORT',
        'Vendas: ' || v_total_sales || ' | Clientes: ' || v_customers_updated,
        NOW()
    );

    RAISE NOTICE 'Relatório %: Vendas R$ % | Pontos atualizados para % clientes.',
                 v_yesterday, v_total_sales, v_customers_updated;

EXCEPTION
    WHEN OTHERS THEN
        INSERT INTO audit_logs (action, details, created_at)
        VALUES ('REPORT_ERROR', SQLERRM, NOW());
        RAISE EXCEPTION 'Erro: %', SQLERRM;
END;
$$ LANGUAGE plpgsql;

-- nao consegui fazer com o CRON
-- SELECT cron.schedule('0 6 * * *', 'CALL daily_ecommerce_report()');